# 03 — Colab single-image debug

Mirrors `02_inference.ipynb` but for iteration: load the model **once**, then re-run the pipeline cell as many times as you want with different configs / prompts / images. ~2 min cold start, then ~30s per image.

**Runtime:** Runtime → Change runtime type → T4 GPU (free tier works for 4-bit 7B).

**Optional:** to load a LoRA adapter from HF Hub, add an `HF_TOKEN` secret via the key icon in the left sidebar (only needed for private repos).

In [ ]:
# 1. Install GPU stack. Colab already has torch + CUDA.
%pip -q install --upgrade unsloth unsloth_zoo
%pip -q install --no-deps trl peft accelerate bitsandbytes
%pip -q install pyyaml opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 MB 8.4 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 23.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 878.6 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 66.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.2 MB/s eta 0:00:000:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
GITHUB_REPO     = "https://github.com/dhodyrev/handwritten-to-data.git"
HF_ADAPTER_REPO = "dkhodyriev/htr-qwen25vl-transcribe-lora"  # set None for base model
CONFIG          = "configs/pipeline.yaml"                     # or configs/pipeline_p1.yaml

In [ ]:
# 2. Clone repo (or hard-pull latest main if already cloned) and install htr.
import os, subprocess, sys
REPO_DIR = "/content/repo"
GIT = ["git", "-C", REPO_DIR]
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
else:
    # Discard any local state and jump to the freshest origin/main.
    subprocess.run(GIT + ["fetch", "origin", "main"], check=True)
    subprocess.run(GIT + ["reset", "--hard", "origin/main"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
# Editable install points at the source dir, so a pull alone refreshes pure-Python
# code; re-running keeps deps in sync if pyproject changed.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], check=True)
print("HEAD:", subprocess.check_output(GIT + ["rev-parse", "--short", "HEAD"]).decode().strip())

In [ ]:
# 3. Optional HF login (only needed if the adapter repo is private).
if HF_ADAPTER_REPO:
    try:
        from google.colab import userdata
        from huggingface_hub import login
        login(token=userdata.get("HF_TOKEN"))
        print("hf login OK")
    except Exception as e:
        print(f"hf login skipped ({e}); proceeding anonymously (only works if adapter is public)")

In [ ]:
# 4. Load model ONCE — keep this cell's variables alive while iterating below.
from htr.backend import DEFAULT_MODEL, load_model
handle = load_model(DEFAULT_MODEL, adapter_path=HF_ADAPTER_REPO)
print("model ready:", handle.name, "+ adapter" if handle.adapter_path else "(base)")

In [ ]:
# 5. Get one image. Pick A or B.
#
# A) Upload from your Mac:
import os
from google.colab import files
uploaded = files.upload()                          # opens file picker
# files.upload() saves into the *current* working dir (cell 3 chdir'd to the
# repo), so resolve the real path instead of assuming /content/.
IMAGE_PATH = os.path.abspath(next(iter(uploaded)))
SOURCE = "unknown"                                 # or school / university / dictation / archive

# B) Or pull a sample page from the HF dataset (uncomment, comment out A):
# from htr.data import materialize_image
# IMAGE_PATH = materialize_image("train", "some_page_filename.jpg")
# SOURCE = "school"

print("image:", IMAGE_PATH)

In [ ]:
# DEBUG: confirm Qwen2.5-VL coordinate convention on THIS page.
# Reuses the model from cell 4 (`handle`) and IMAGE_PATH from cell 5.
import torch
from htr import prompts as P
from htr.pipeline import parse_json
from PIL import Image

model, processor = handle.model, handle.processor
img = Image.open(IMAGE_PATH).convert("RGB")
print("original  W x H :", img.size)

messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": P.QWEN_BLOCK_DETECT_PROMPT},
]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[img], return_tensors="pt", padding=True).to(model.device)

# image_grid_thw is in 14px patch units; smart-resized frame = grid * 14.
t, gh, gw = inputs["image_grid_thw"][0].tolist()
res_w, res_h = gw * 14, gh * 14
print("grid_thw       :", (t, gh, gw), "-> resized W x H (px):", (res_w, res_h))

with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=1024,
                         do_sample=False, temperature=0.0, use_cache=True)
raw = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                             skip_special_tokens=True)[0]
print("\nRAW detect output:\n", raw)

data = parse_json(raw)
blocks = data.get("blocks", data) if isinstance(data, dict) else (data or [])
print("\nbbox coord magnitudes:")
for b in blocks:
    bb = b.get("bbox_2d", b.get("bbox")) if isinstance(b, dict) else b
    if bb:
        print("  ", bb, "| max_x =", max(bb[0], bb[2]), " max_y =", max(bb[1], bb[3]))
print(f"\nVERDICT: coords ~<=1000 => 0-1000 normalized (current code is right).")
print(f"         coords ~ up to {res_w}x{res_h} => ABSOLUTE pixels (Qwen2.5 native; code is wrong).")

In [ ]:
# DEBUG: inspect line-detection inside the detected text_block.
import base64
from htr import prompts as P
from htr.backend import qwen_call_geo
from htr.pipeline import parse_json, _items_from_payload, _parse_lines
from htr.image_ops import encode_jpeg
from PIL import Image

img = Image.open(IMAGE_PATH).convert("RGB")
bbox = [0, 594, 3120, 4077]          # the text_block from detection
img_w, img_h = img.size
x1, y1, x2, y2 = bbox
mx, my = int((x2 - x1) * 0.03), int((y2 - y1) * 0.03)
cx1, cy1 = max(0, x1 - mx), max(0, y1 - my)
cx2, cy2 = min(img_w, x2 + mx), min(img_h, y2 + my)
crop = img.crop((cx1, cy1, cx2, cy2))
crop_w, crop_h = crop.size
b64 = base64.b64encode(encode_jpeg(crop)).decode()

raw, (sw, sh) = qwen_call_geo(b64, P.QWEN_LINE_DETECT_PROMPT)
print("crop WxH:", (crop_w, crop_h), "| resized frame:", (sw, sh))
print("\nRAW line-detect output (first 1500 chars):\n", raw[:1500])
print("\nparsed json type:", type(parse_json(raw)).__name__)
print("items found     :", len(_items_from_payload(raw, "lines", "blocks")))
lines = _parse_lines(raw, crop_w, crop_h, sw, sh, cx1, cy1)
print("parsed line bboxes:", len(lines))
for lb in lines[:6]:
    print("  ", lb)

In [ ]:
# TEST: full pipeline with the new multi-line + per-line-band path.
# Reloads htr modules (keeping the loaded model) so src/ pulls take effect.
import importlib, htr.backend, htr.pipeline, htr.prompts
_saved = htr.backend._HANDLE
importlib.reload(htr.backend); htr.backend._HANDLE = _saved   # keep model live
importlib.reload(htr.prompts); importlib.reload(htr.pipeline)
from htr.config import load_pipeline_config
from htr.pipeline import run_pipeline
from pathlib import Path

cfg = load_pipeline_config(CONFIG)
result = run_pipeline(IMAGE_PATH, Path(IMAGE_PATH).stem, SOURCE, cfg)

print(f"\n{len(result['regions'])} regions")
for i, r in enumerate(result["regions"]):
    print(f"[{i:2d}] {r['bbox']}  {r.get('transcription','')[:80]}")

print("\n--- page text (what PageCER sees) ---")
print("\n".join(r.get("transcription", "") for r in result["regions"]))

In [ ]:
# 6. Run the pipeline. Re-run this cell after tweaking CONFIG / SOURCE / prompts.
import importlib, htr.pipeline, htr.prompts
importlib.reload(htr.prompts); importlib.reload(htr.pipeline)  # pick up prompt edits without restart
from htr.config import load_pipeline_config
from htr.pipeline import run_pipeline
from pathlib import Path

cfg = load_pipeline_config(CONFIG)
result = run_pipeline(IMAGE_PATH, Path(IMAGE_PATH).stem, SOURCE, cfg)

print(f"{len(result['regions'])} regions\n")
for i, r in enumerate(result["regions"]):
    print(f"[{i}] {r.get('type','?'):12s} bbox={r['bbox']}")
    print(f"     {r.get('transcription','')[:200]}")
    print()

In [ ]:
# 7. Visualise: draw bboxes over the image.
from PIL import Image, ImageDraw
img = Image.open(IMAGE_PATH).convert("RGB")
W, H = img.size
draw = ImageDraw.Draw(img)
for i, r in enumerate(result["regions"]):
    x0, y0, x1, y1 = r["bbox"]
    # bboxes are normalised to 0–1000
    box = [x0 * W / 1000, y0 * H / 1000, x1 * W / 1000, y1 * H / 1000]
    draw.rectangle(box, outline="red", width=3)
    draw.text((box[0] + 4, box[1] + 4), str(i), fill="red")
img.thumbnail((1200, 1200))
img

## Iterating

- **Prompt tweak:** edit `src/htr/prompts.py` in the file browser, then re-run cell 6 (the `importlib.reload` line picks it up — no kernel restart).
- **Config tweak:** edit `configs/pipeline.yaml` (or switch `CONFIG` to `pipeline_p1.yaml`), re-run cell 6.
- **Different image:** re-run cell 5 (option A re-uploads), then cell 6.
- **Different source routing:** change `SOURCE` in cell 5, re-run cell 6.

Model stays loaded in cell 4's `handle` — only restart that cell if you change `HF_ADAPTER_REPO` or the base model.